# 🚀 Asynchronous Web Scraper Pipeline — Production Workflow

This notebook orchestrates a **production-grade asynchronous web scraping pipeline** using the refactored, highly resilient `data_scraper_engine` interface.
All extracted data is dynamically routed based on extraction methods, protected against IP rate limits via random jitter, and immediately persisted to disk via incremental checkpoints to avoid data loss.

---

---

### 📜 The Scraper Configuration Ledger (Schema)

| Key | Type | Description | Available Options | Example |
|:---|:---|:---|:---|:---|
| `target_urls` | `List[str]` | List of target web page URLs to scrape. | List of absolute URLs | `["https://quotes.toscrape.com/page/1/"]` |
| `output_filename` | `str` | Base path/filename on disk where data is saved. | Any valid relative or absolute path | `"Exports/clean_dataset_v1"` |
| `export_format` | `str` | Target export format. Temp CSV is used for Excel checkpoints. | `"csv"`, `"jsonl"`, `"excel"` | `"csv"` |
| `container_selector` | `str` | Relative repeating parent DOM element boundaries. | Any valid CSS selector | `".quote"` |
| `fields_to_scrape` | `Dict` | Mapping of headers to selectors and extraction methods. | Dict of keys to CSS query or config schema | See details below |

### 🎯 Extraction Routing Methods (`fields_to_scrape` Schema)
To prevent **silent data loss**, you must route fields to their correct extraction methods based on the expected cardinality:
* `"method": "get"` (or `"type": "str"`) ➡️ Extracts the first matching element's text (ideal for titles, authors, prices, dates).
* `"method": "getall"` (or `"type": "list"`) ➡️ Extracts all matching elements as a list of strings (ideal for categories, tag arrays, image galleries).
* **Fallback Strategy**: Standard raw CSS selector strings (e.g., `".title::text"`) are fully supported and automatically default to the single-string `"get"` method.

### 🚫 Strict Rules
1. **Headless & Stealth Active**: The engine utilizes `AsyncStealthySession(headless=True, adaptive=True, solve_cloudflare=True)` to solve Turnstile and captcha challenges and adapt to minor changes in target page structures.
2. **Temporary File Safeguard**: When exporting to `excel`, a temporary CSV file (`{filename}_temp_checkpoint.csv`) is created on disk during runtime to write progress and is converted/cleaned up upon successful compilation.

### 📋 Blank Configuration Template

Copy this block to configure a fresh scraper run:

```python
SCRAPER_CONFIG = {
    "target_urls": [
        "https://quotes.toscrape.com/page/1/",
        "https://quotes.toscrape.com/page/2/"
    ],
    "output_filename": "Exports/scraped_data",
    "export_format": "csv",
    "container_selector": ".item-container",
    "fields_to_scrape": {
        "title": {"selector": ".title::text", "method": "get"},
        "price": {"selector": ".price::text", "method": "get"},
        "tags": {"selector": ".tags .tag::text", "method": "getall"}
    }
}
```

---
## ⚙️ Step 1: Web Scraper Configuration
Customize the configuration block below for your target pages and schema mappings.

In [1]:
# ══════════════════════════════════════════════════════════════════════
#  Pipeline Configuration
# ══════════════════════════════════════════════════════════════════════

SCRAPER_CONFIG = {
    # Batch Target URLs
    "target_urls": [
        "https://quotes.toscrape.com/page/1/",
        "https://quotes.toscrape.com/page/2/"
    ],
    # Checkpoint File Output Base Name (no extension)
    "output_filename": "Exports/clean_dataset_v1",
    # Final export format. Options: "csv", "jsonl", "excel"
    "export_format": "csv",  
    # Repeating CSS container boundaries
    "container_selector": ".quote",
    # CSS Selectors with explicit extraction routing methods
    "fields_to_scrape": {
        "author": {"selector": ".author::text", "method": "get"},
        "quote_text": {"selector": ".text::text", "method": "get"},
        "tags": {"selector": ".tags .tag::text", "method": "getall"}
    }
}

---
## 🕷️ Step 2: Asynchronous Scraper Pipeline Execution
Running this block initiates the isolated subprocess. Data will be saved continuously to the disk checkpoint.

In [2]:
# ══════════════════════════════════════════════════════════════════════
#  Pipeline Execution & Consolidation
# ══════════════════════════════════════════════════════════════════════

from data_scraper_engine import execute_pipeline
df = execute_pipeline(SCRAPER_CONFIG)

scraped_url,author,quote_text,tags
https://quotes.toscrape.com/page/1/,Albert Einstein,“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”,"[change, deep-thoughts, thinking, world]"
https://quotes.toscrape.com/page/1/,J.K. Rowling,"“It is our choices, Harry, that show what we truly are, far more than our abilities.”","[abilities, choices]"
https://quotes.toscrape.com/page/1/,Albert Einstein,“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”,"[inspirational, life, live, miracle, miracles]"
https://quotes.toscrape.com/page/1/,Jane Austen,"“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”","[aliteracy, books, classic, humor]"
https://quotes.toscrape.com/page/1/,Marilyn Monroe,"“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”","[be-yourself, inspirational]"


---
## 🛠️ Step-by-Step Guide: How to Scrape a New Website

Follow these clear engineering steps to configure and run the scraper on any new target website:

### Step 1: DOM Inspection (Identify Repeating Containers)
Open your browser and navigate to the target site. Press `F12` to open the Developer Tools, select the **Elements** tab, and locate the repeating card/block containing the data items (e.g., product boxes, articles, user reviews):
* This repeating element class or tag will serve as your **`container_selector`** (e.g., `".quote"`, `"div.product-card"`, `"article.post-item"`).

### Step 2: Child Element CSS Mapping
Inspect the elements inside a single repeating container block to find the relative child elements you want to extract:
* Target the child element classes or tags (e.g., `.author::text`, `span.price::text`, `.tags .tag::text`).
* Verify if a field has a single text occurrence or multiple values (e.g., a product only has one title, but can have a list of tag names).

### Step 3: Define Your Extraction Schema (`fields_to_scrape`)
Map each target key to its appropriate extraction method to ensure zero data loss:
1. **Single String Value**: Map to `"method": "get"` (or `"type": "str"`). Example:
   `"author": {"selector": ".author::text", "method": "get"}`
2. **Multiple Values (Array/List)**: Map to `"method": "getall"` (or `"type": "list"`). Example:
   `"tags": {"selector": ".tags .tag::text", "method": "getall"}`
3. **Backward Compatible String**: You can also just pass the raw CSS selector string directly, and the engine will automatically default to `"get"` mode:
   `"quote_text": ".text::text"`

### Step 4: Map Batch Parameters & Target URLs
Compile a list of target page URLs (`target_urls`) to scrape in a single batch, and specify your base output filename and format (`"csv"`, `"jsonl"`, or `"excel"`).

### Step 5: Load and Run the Pipeline
Update your configuration block in **Step 1 (Configuration)** and run the execution cell in **Step 2 (Execution)**. The engine isolates the Tornado loop, handles bot-stealth, executes safe sequential scrapes with random jitter, writes immediate checkpoints to disk to prevent RAM bloat, and parses list-based fields natively!